# Dataset-2 visual map with COLMAP

This notebook builds a **sparse Structure-from-Motion (SfM) visual map** from the ordinary video frames in dataset-2. It does not need ARKit poses. COLMAP estimates the reference camera trajectory, camera calibration, and a 3D landmark point cloud from image overlap.

The result is a visual map suitable for image retrieval and 6-DoF visual localization. Its coordinate system and scale are arbitrary; apply one measured distance or a floor-plan alignment if metric coordinates are needed.

## One-time installation

Install the COLMAP command-line application, then restart the kernel. On macOS, run `brew install colmap`; with Conda, run `conda install -c conda-forge colmap`. This notebook deliberately uses the COLMAP executable rather than ARKit or PyCOLMAP.

This configuration uses every fifth extracted frame (about 2 frames per second). The first 1-fps reconstruction only registered five images. If matching is too slow, raise `FRAME_STRIDE`; if the reconstruction is fragmented, lower it. Use sharp frames with approximately 0.5–2 m motion between selected images.

In [13]:
from pathlib import Path
import shutil
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src' / 'indoor_vpr').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Could not find the indoor-VPR project root.')
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'dataset-2'
# Build the map from one traversal. Keep the other traversal for later localization.
REFERENCE_FRAMES = DATASET_ROOT / 'frames-IMG_3609'
QUERY_FRAMES = DATASET_ROOT / 'frames-IMG_3610'
MAP_ROOT = PROJECT_ROOT / 'outputs' / 'dataset-2_visual_map'
SAMPLED_IMAGES = MAP_ROOT / 'reference_images'
DATABASE_PATH = MAP_ROOT / 'database.db'
SPARSE_ROOT = MAP_ROOT / 'sparse'
EXPORT_ROOT = MAP_ROOT / 'export'

# The first 1-fps attempt registered only five frames. Use denser 2-fps sampling.
FRAME_STRIDE = 5  # extracted frames are about 10 fps; use about 2 fps for this traversal
SEQUENTIAL_OVERLAP = 20  # selected temporal neighbours considered for matching
USE_GPU = False  # portable default; set True only for a CUDA-enabled COLMAP build

if not REFERENCE_FRAMES.is_dir():
    raise FileNotFoundError(f'Missing reference images: {REFERENCE_FRAMES}')
if shutil.which('colmap') is None:
    raise RuntimeError('COLMAP is not on PATH. Install it, restart the kernel, and rerun this cell.')

all_reference_images = sorted(REFERENCE_FRAMES.glob('*.jpg'))
if not all_reference_images:
    raise RuntimeError(f'No JPEG frames found in {REFERENCE_FRAMES}')
selected_images = all_reference_images[::FRAME_STRIDE]
print(f'Reference frames: {len(all_reference_images):,}')
print(f'Selected map images: {len(selected_images):,}')
print(f'Output directory: {MAP_ROOT}')

Reference frames: 9,318
Selected map images: 1,864
Output directory: /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map


## Prepare map images

COLMAP is given a compact directory containing only the sampled reference images. The files are copied once, so the original dataset remains unchanged. Set `REBUILD_SAMPLES` to `True` only after changing `FRAME_STRIDE` or the source traversal.

In [14]:
REBUILD_SAMPLES = True

expected_names = {image.name for image in selected_images}
existing_names = {image.name for image in SAMPLED_IMAGES.glob('*.jpg')} if SAMPLED_IMAGES.exists() else set()
if REBUILD_SAMPLES or existing_names != expected_names:
    if SAMPLED_IMAGES.exists():
        shutil.rmtree(SAMPLED_IMAGES)
    SAMPLED_IMAGES.mkdir(parents=True)
    for index, image in enumerate(selected_images, start=1):
        shutil.copy2(image, SAMPLED_IMAGES / image.name)
        if index % 100 == 0 or index == len(selected_images):
            print(f'Copied {index:,}/{len(selected_images):,} images')
else:
    print(f'Reusing {len(existing_names):,} sampled images in {SAMPLED_IMAGES}')

Copied 100/1,864 images
Copied 200/1,864 images
Copied 300/1,864 images
Copied 400/1,864 images
Copied 500/1,864 images
Copied 600/1,864 images
Copied 700/1,864 images
Copied 800/1,864 images
Copied 900/1,864 images
Copied 1,000/1,864 images
Copied 1,100/1,864 images
Copied 1,200/1,864 images
Copied 1,300/1,864 images
Copied 1,400/1,864 images
Copied 1,500/1,864 images
Copied 1,600/1,864 images
Copied 1,700/1,864 images
Copied 1,800/1,864 images
Copied 1,864/1,864 images


## Run sparse SfM

Run these cells in order. They create a feature database, match temporally neighbouring frames, and estimate the sparse 3D reconstruction. `OVERWRITE_RECONSTRUCTION=True` removes only this notebook's previous COLMAP intermediates under `outputs/dataset-2_visual_map`; it never changes the dataset frames.

In [15]:
OVERWRITE_RECONSTRUCTION = True

if OVERWRITE_RECONSTRUCTION:
    for path in (DATABASE_PATH, SPARSE_ROOT, EXPORT_ROOT):
        if path.is_dir():
            shutil.rmtree(path)
        elif path.exists():
            path.unlink()

MAP_ROOT.mkdir(parents=True, exist_ok=True)
SPARSE_ROOT.mkdir(parents=True, exist_ok=True)
gpu_flag = '1' if USE_GPU else '0'

def run_colmap(*arguments):
    command = ['colmap', *map(str, arguments)]
    print(' '.join(command))
    subprocess.run(command, check=True)

if not DATABASE_PATH.exists():
    run_colmap(
        'feature_extractor',
        '--database_path', DATABASE_PATH,
        '--image_path', SAMPLED_IMAGES,
        '--ImageReader.single_camera', '1',
        # A constrained camera model prevents unstable distortion estimates in corridors.
        '--ImageReader.camera_model', 'SIMPLE_RADIAL',
        # Approximate iPhone-video focal length after portrait rotation.
        '--ImageReader.default_focal_length_factor', '0.8',
        # COLMAP 4.x moved this option from SiftExtraction to FeatureExtraction.
        '--FeatureExtraction.use_gpu', gpu_flag,
        '--SiftExtraction.max_num_features', '16384',
    )
else:
    print(f'Reusing feature database: {DATABASE_PATH}')

colmap feature_extractor --database_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/database.db --image_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/reference_images --ImageReader.single_camera 1 --ImageReader.camera_model SIMPLE_RADIAL --ImageReader.default_focal_length_factor 0.8 --FeatureExtraction.use_gpu 0 --SiftExtraction.max_num_features 16384


W20260904 16:15:22.232560 0x1ed1f9e80 feature_extraction.cc:445] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I20260904 16:15:22.233178 0x16cfef000 feature_extraction.cc:494] === Feature extraction ===
I20260904 16:15:22.233263 0x16d70b000 sift.cc:763] Creating SIFT CPU feature extractor
I20260904 16:15:22.233265 0x16d797000 sift.cc:763] Creating SIFT CPU feature extractor
I20260904 16:15:22.233273 0x16d823000 sift.cc:763] Creating SIFT CPU feature extractor
I20260904 16:15:22.233282 0x16d93b000 sift.cc:763] Creating SIFT CPU feature extractor
I20260904 16:15:22.233287 0x16d9c7000 sift.cc:763] Creating SIFT CPU feature extractor
I20260904 16:15:22.233297

In [16]:
# Run only once per database. Delete database.db or set OVERWRITE_RECONSTRUCTION=True to rerun.
run_colmap(
    'sequential_matcher',
    '--database_path', DATABASE_PATH,
    '--SequentialMatching.overlap', SEQUENTIAL_OVERLAP,
    # COLMAP 4.x moved this option from SiftMatching to FeatureMatching.
    '--FeatureMatching.use_gpu', gpu_flag,
)

if not any(SPARSE_ROOT.iterdir()):
    run_colmap(
        'mapper',
        '--database_path', DATABASE_PATH,
        '--image_path', SAMPLED_IMAGES,
        '--output_path', SPARSE_ROOT,
        '--Mapper.ba_refine_principal_point', '0',
        # Keep radial distortion fixed: long indoor corridors otherwise permit a degenerate fit.
        '--Mapper.ba_refine_extra_params', '0',
        # Dataset-2 has low-texture/repetitive indoor sections. These relaxed
        # registration thresholds retain valid frames that the default mapper rejects.
        '--Mapper.init_min_num_inliers', '50',
        '--Mapper.abs_pose_min_num_inliers', '15',
        '--Mapper.abs_pose_min_inlier_ratio', '0.10',
    )
else:
    print(f'Reusing sparse models in {SPARSE_ROOT}')

colmap sequential_matcher --database_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/database.db --SequentialMatching.overlap 20 --FeatureMatching.use_gpu 0


I20260904 16:27:07.893211 0x16bca3000 feature_matching.cc:195] === Feature matching & geometric verification ===
I20260904 16:27:08.032916 0x16be47000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032926 0x16bed3000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032931 0x16bf5f000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032906 0x16bdbb000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032962 0x16c103000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032969 0x16c18f000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032975 0x16c21b000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032956 0x16c077000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032951 0x16bfeb000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032988 0x16c2a7000 sift.cc:1565] Creating SIFT CPU feature matcher
I20260904 16:27:08.032906 0

colmap mapper --database_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/database.db --image_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/reference_images --output_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/sparse --Mapper.ba_refine_principal_point 0 --Mapper.ba_refine_extra_params 0


I20260904 16:28:49.109097 0x1ed1f9e80 incremental_pipeline.cc:278] Loading database
I20260904 16:28:49.109472 0x1ed1f9e80 database_cache.cc:72] Loading rigs...
I20260904 16:28:49.109492 0x1ed1f9e80 database_cache.cc:82]  1 in 0.000s
I20260904 16:28:49.109496 0x1ed1f9e80 database_cache.cc:90] Loading cameras...
I20260904 16:28:49.109502 0x1ed1f9e80 database_cache.cc:108]  1 in 0.000s
I20260904 16:28:49.109504 0x1ed1f9e80 database_cache.cc:116] Loading frames...
I20260904 16:28:49.110292 0x1ed1f9e80 database_cache.cc:126]  1864 in 0.001s
I20260904 16:28:49.110295 0x1ed1f9e80 database_cache.cc:134] Loading matches...
I20260904 16:28:49.113982 0x1ed1f9e80 database_cache.cc:139]  4650 in 0.004s
I20260904 16:28:49.113989 0x1ed1f9e80 database_cache.cc:147] Loading images...
I20260904 16:28:49.144453 0x1ed1f9e80 database_cache.cc:241]  1864 in 0.030s (connected 1678, loaded 1678)
I20260904 16:28:49.144496 0x1ed1f9e80 database_cache.cc:255] Loading pose priors...
I20260904 16:28:49.144502 0x1ed

## Inspect and export the map

COLMAP may produce more than one model if the trajectory becomes disconnected. This cell selects the model registering the most cameras, prints its reconstruction statistics, and exports a PLY point cloud. Open the PLY in MeshLab or CloudCompare; open the selected model directly in the COLMAP GUI for camera poses and image observations.

In [18]:
models = sorted(path for path in SPARSE_ROOT.iterdir() if path.is_dir() and (path / 'images.bin').exists())
if not models:
    raise RuntimeError('No sparse model was created. See the troubleshooting cell below.')

def registered_image_count(model):
    result = subprocess.run(
        ['colmap', 'model_analyzer', '--path', str(model)],
        check=True, text=True, capture_output=True,
    )
    for line in result.stdout.splitlines():
        if line.strip().startswith('Registered images'):
            return int(line.rsplit(':', 1)[1].strip()), result.stdout
    return 0, result.stdout

analyses = [(registered_image_count(model), model) for model in models]
(count, analysis), BEST_MODEL = max(analyses, key=lambda item: item[0][0])
print(f'Selected model: {BEST_MODEL}')
print(analysis)
if count < 0.10 * len(selected_images):
    print(
        f'WARNING: only {count}/{len(selected_images)} images registered. '
        'This is not yet a usable visual map. Rebuild from the earlier cells with '
        'FRAME_STRIDE=5, REBUILD_SAMPLES=True, and OVERWRITE_RECONSTRUCTION=True.'
    )

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
POINT_CLOUD_PATH = EXPORT_ROOT / 'dataset2_sparse_map.ply'
run_colmap(
    'model_converter',
    '--input_path', BEST_MODEL,
    # COLMAP 4.x expects the PLY filename here, not an output directory.
    '--output_path', POINT_CLOUD_PATH,
    '--output_type', 'PLY',
)
print(f'PLY point cloud: {POINT_CLOUD_PATH}')
print(f'COLMAP GUI: colmap gui  # then File > Import model and choose {BEST_MODEL}')

Selected model: /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/sparse/0

colmap model_converter --input_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/sparse/0 --output_path /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/export/dataset2_sparse_map.ply --output_type PLY
PLY point cloud: /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/export/dataset2_sparse_map.ply
COLMAP GUI: colmap gui  # then File > Import model and choose /Users/armin/Documents/indoor-VPR/outputs/dataset-2_visual_map/sparse/0


## Troubleshooting and next step

- **Few registered images / disconnected map:** reduce `FRAME_STRIDE` (for example 5), reject blurred frames, and increase `SEQUENTIAL_OVERLAP` to 20–30.
- **Repeated hallway false matches:** use `vocab_tree_matcher` or spatial verification after the sequential reconstruction is stable; do not begin with exhaustive matching over all images.
- **Wrong scale:** SfM cannot recover an absolute scale from this monocular video. Scale the model after reconstruction with a measured hallway distance.
- **Use the second video:** retain `frames-IMG_3610` as a query traversal. Extract its features and use HLoc's `localize_sfm` or COLMAP image registration against `BEST_MODEL`; this repository's ARKit HLoc notebook is not applicable unchanged because its map stage assumes known ARKit poses.
- **Dense model:** once the sparse model is sound, run COLMAP's image undistorter, patch-match stereo, and stereo fusion. Dense reconstruction is optional for visual localization; the sparse landmarks and camera poses are the map needed first.